In [1]:
!pip install scikit-learn xgboost

In [2]:
import numpy as np
import pandas as pd

X = np.load("/kaggle/input/notebooks/navishaaagarwaal/05-proposed-model/X_features.npy")
y = np.load("/kaggle/input/notebooks/navishaaagarwaal/05-proposed-model/y_labels.npy")

print(X.shape)

(11902, 3077)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
from sklearn.metrics import classification_report, roc_auc_score

def evaluate_model(model, name):

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    roc = roc_auc_score(y_test, y_prob)

    print("\n", "="*40)
    print(name)
    print("="*40)
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc)

    return roc

In [7]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

roc_lr = evaluate_model(lr, "Logistic Regression")


Logistic Regression
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      2083
           1       0.33      0.47      0.39       298

    accuracy                           0.81      2381
   macro avg       0.62      0.67      0.64      2381
weighted avg       0.84      0.81      0.83      2381

ROC-AUC: 0.7455584839883106


In [8]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    n_jobs=-1
)

roc_rf = evaluate_model(rf, "Random Forest")


Random Forest
              precision    recall  f1-score   support

           0       0.88      1.00      0.94      2083
           1       0.92      0.04      0.08       298

    accuracy                           0.88      2381
   macro avg       0.90      0.52      0.51      2381
weighted avg       0.88      0.88      0.83      2381

ROC-AUC: 0.8008671991545494


In [9]:
neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1
)

roc_xgb = evaluate_model(xgb, "XGBoost")


XGBoost
              precision    recall  f1-score   support

           0       0.92      0.94      0.93      2083
           1       0.51      0.46      0.48       298

    accuracy                           0.88      2381
   macro avg       0.71      0.70      0.70      2381
weighted avg       0.87      0.88      0.87      2381

ROC-AUC: 0.8308494459784707


In [10]:
mlp = MLPClassifier(
    hidden_layer_sizes=(512,128),
    max_iter=200,
    batch_size=64,
    learning_rate_init=0.001,
    early_stopping=True
)

roc_mlp = evaluate_model(mlp, "MLP Neural Network")


MLP Neural Network
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      2083
           1       0.69      0.28      0.40       298

    accuracy                           0.89      2381
   macro avg       0.80      0.63      0.67      2381
weighted avg       0.88      0.89      0.87      2381

ROC-AUC: 0.7900171409975932


In [11]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "MLP"
    ],
    "ROC-AUC":[
        roc_lr,
        roc_rf,
        roc_xgb,
        roc_mlp
    ]
})

results

,Model,ROC-AUC
0,Logistic Regression,0.745558
1,Random Forest,0.800867
2,XGBoost,0.830849
3,MLP,0.790017
